# REBA — Entrenamiento YOLOv8n → ONNX

Cuaderno preparado para **Google Colab** con GPU **T4**.

## Objetivo
1. Instalar Ultralytics YOLOv8
2. Montar / subir el dataset generado por `dataset_builder.py`
3. Entrenar `yolov8n` durante 50–100 épocas
4. Exportar `best.onnx` para el servidor REBA (`models/best.onnx`)

## Clases
`plastico`, `tela`, `organico`, `unicel`, `metal`

Documentación: `docs/ai_pipeline.md`

## 1. Comprobar GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Instalar dependencias

In [ ]:
!pip -q install ultralytics onnx onnxruntime opencv-python-headless
from ultralytics import YOLO
print('Ultralytics OK')

## 3. Preparar dataset

Opciones:
- **A)** Subir un ZIP del dataset (`images/`, `labels/`, `data.yaml`) a Colab
- **B)** Montar Google Drive y apuntar `DATA_YAML` a la ruta del `data.yaml`

Estructura esperada (la genera `dataset_builder.py`):
```
REBA/
  data.yaml
  images/train ...
  images/val ...
  labels/train ...
  labels/val ...
```

In [ ]:
# --- Opción B: Google Drive (descomenta si aplica) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_YAML = '/content/drive/MyDrive/REBA/datasets/REBA/data.yaml'

# --- Opción A: subir ZIP y descomprimir ---
from google.colab import files
import zipfile
from pathlib import Path

UPLOAD = True  # pon False si ya montaste Drive y definiste DATA_YAML

if UPLOAD:
    print('Sube el ZIP del dataset REBA…')
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    out = Path('/content/REBA_dataset')
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall(out)
    # Buscar data.yaml
    yamls = list(out.rglob('data.yaml'))
    assert yamls, 'No se encontró data.yaml en el ZIP'
    DATA_YAML = str(yamls[0])

print('DATA_YAML =', DATA_YAML)
print(Path(DATA_YAML).read_text())

## 4. Parámetros de entrenamiento (editables)

| Parámetro | Default | Notas |
|-----------|---------|-------|
| `EPOCHS` | 80 | Rango recomendado 50–100 |
| `IMGSZ` | 640 | Coincide con el servidor |
| `BATCH` | 16 | Baja a 8 si hay OOM en T4 |
| `MODEL` | yolov8n.pt | nano = rápido en CPU del PC |

In [ ]:
EPOCHS = 80          # 50–100 recomendado
IMGSZ = 640
BATCH = 16
MODEL = 'yolov8n.pt' # n=nano, s=small
PROJECT = '/content/REBA_runs'
NAME = 'waste_yolov8n'

model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name=NAME,
    pretrained=True,
    patience=20,
    device=0,          # GPU
    workers=2,
)
print('Entrenamiento finalizado')

## 5. Validar el mejor checkpoint

In [ ]:
best_pt = f'{PROJECT}/{NAME}/weights/best.pt'
print('Best weights:', best_pt)
best = YOLO(best_pt)
metrics = best.val(data=DATA_YAML)
print(metrics)

## 6. Exportar a ONNX

El servidor REBA carga `models/best.onnx` con **onnxruntime** (CPU).
`opset=12` e `imgsz=640` son compatibles con `server/inference.py`.

In [ ]:
export_path = best.export(format='onnx', imgsz=640, opset=12, simplify=True)
print('ONNX exportado en:', export_path)

# Descargar al PC local
from google.colab import files
files.download(export_path)
print('Guarda el archivo descargado como: REBA/models/best.onnx')

## 7. Prueba rápida de inferencia ONNX (opcional)

In [ ]:
onnx_model = YOLO(export_path)
# Sustituye por una imagen de prueba del dataset si quieres
# onnx_model.predict(source='ruta/imagen.jpg', save=True, conf=0.45)
print('Listo. Copia best.onnx a models/ y arranca el servidor REBA.')